# LegalGPT: sentence-role classifier experiments (InLegalBERT), on a Colab GPU

Runs the context ablation (sentence only / + position / + neighbours / + both) as 5-fold
cross-validation grouped by judgment, then the BiLSTM sequence model on the same folds and its
ensemble with the best fine-tuned model. Settings are identical to the laptop runs.

**Before you start:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → Run all.
When asked, upload `legalgpt_rrl.zip` (from `data/colab/` on the laptop).
At the end, `rrl_results.zip` downloads; send it back.

Time on an L4: about 30–40 minutes in total.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Upload the package and install

In [ ]:
import os, zipfile
from google.colab import files
os.chdir('/content')
if not os.path.exists('/content/legaltech/backend/app'):
    up = files.upload()  # choose legalgpt_rrl.zip
    name = next(iter(up))
    os.makedirs('/content/legaltech', exist_ok=True)
    zipfile.ZipFile(name).extractall('/content/legaltech')
os.chdir('/content/legaltech')
print(sorted(os.listdir('data/annotation_store/versions')))

In [ ]:
# Colab's own CUDA build of PyTorch is kept; only the libraries the scripts need are pinned
!pip -q install transformers==5.18.0 pydantic-settings==2.15.0 python-dotenv==1.2.4
!pip -q uninstall -y torchao

## 2. Context ablation: four 5-fold cross-validation runs

Same settings as the laptop: max length 256, all layers trained, 6 epochs, bf16, seed 13.
Every held-out prediction is saved (for paired significance tests and the ensemble).

In [ ]:
VERSION = 'v0.3'
BASE = f'python backend/scripts/train_rrl_classifier.py --version {VERSION} --max-length 256 --freeze-layers 0 --epochs 6 --bf16 --cv 5 --save-predictions'
RUNS = {
    'abl-sentence':   '',
    'abl-position':   '--context --context-mode position',
    'abl-neighbours': '--context --context-mode neighbours',
    'abl-full':       '--context',
}
os.makedirs('data/models', exist_ok=True)
for tag, extra in RUNS.items():
    print(f'=== {tag} ===', flush=True)
    rc = os.system(f'{BASE} {extra} --tag {tag} > data/models/{tag}.log 2>&1')
    os.system(f'grep -E "macro-F1|accuracy" data/models/{tag}.log | tail -3')
    print('exit code', rc, flush=True)

## 3. Sequence model (BiLSTM over the whole judgment) and the ensemble

In [ ]:
rc = os.system(f'python backend/scripts/rrl_sequence_model.py --version {VERSION} '
               f'--folds-from "data/models/rrl_cv/{VERSION}-cv5-abl-full-*.json" '
               f'--ensemble-with "data/models/rrl_cv/{VERSION}-cv5-abl-full-*.json" > data/models/sequence.log 2>&1')
os.system('tail -4 data/models/sequence.log')
print('exit code', rc)

## 4. Download the results

In [ ]:
import glob
with zipfile.ZipFile('/content/rrl_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('data/models/rrl_cv/*.json') + glob.glob('data/models/*.log'):
        z.write(f)
print(os.path.getsize('/content/rrl_results.zip') // 1024, 'KB')
files.download('/content/rrl_results.zip')